In [ ]:
#----------------------------#
# 0.) Idea
#----------------------------#
# Analyze Get submssion deletion boundaries
#

#
import sys
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import Counter
import matplotlib.pyplot as plt
import import_ipynb
import math

In [ ]:
home_path = %pwd
data_path = "/home/pavel/Documents/Keusch_Job/Data/"
plot_path  = "/home/pavel/Documents/Keusch_Job/Tasks/deletion_boundaries/"
subroutines_path = "/home/pavel/Documents/Keusch_Job/Subroutines/"

In [ ]:
#Subroutines
%cd  $subroutines_path
%run "selected_subreddits.ipynb"
%run "data_loader.ipynb"
%run "basic_plots.ipynb"
%cd  $home_path

In [ ]:
S = load_data(data_name="SUBM_2020_01.csv", data_path = data_path)
S.head()

In [ ]:
#----------------------------#
# 1.)  Prob deletion Descriptives
#----------------------------#

In [ ]:
# I would argue to keep automod_filtered, since it is still set by the mods from what I see
#https://www.reddit.com/r/AutoModerator/comments/q11pu/what_is_automoderator/
# "Remove submissions where the title contains certain words or phrases"

In [ ]:
#Some basic Stats
print("Length of Submission Data: " + str(len(S)))
print("Passible removed_by_category outputs" + str(S["removed_by_category"].unique()))
S_grouped = S.groupby(["removed_by_category"]).size().reset_index()
print(S_grouped.to_string())
print("Amount deleted: " + str(S_grouped[0].sum()))

In [ ]:
S["removed"] = 1 
S.loc[S["removed_by_category"].isna(),"removed"] = 0 
S_grouped = S.groupby(["subreddit","removed"]).size().reset_index()
S_grouped.columns = ["subreddit","removed","count"]
print(len(S_grouped.loc[S_grouped["removed"] == 1,]))
print(len(S_grouped.loc[S_grouped["removed"] == 0,]))
print("Each subreddit has deleted at least one submission")

In [ ]:
#Get remove percentages
S_grouped_deletions = S_grouped.loc[S_grouped["removed"] == 0,].merge(S_grouped.loc[S_grouped["removed"] == 1,],
                                                             how="inner", left_on="subreddit",right_on="subreddit")
S_grouped_deletions.columns = ["subreddit","deleted_x","kept","deleted_y","deleted"]
S_grouped_deletions = S_grouped_deletions.drop(columns=['deleted_x', 'deleted_y'])
S_grouped_deletions["overall"] = S_grouped_deletions["kept"]  + S_grouped_deletions["deleted"] 
S_grouped_deletions["rem_perc"] = S_grouped_deletions["deleted"] /  S_grouped_deletions["overall"] 
S_grouped_deletions = S_grouped_deletions.sort_values("rem_perc")

In [ ]:
simple_frequency(S_grouped_deletions, path= plot_path, file_name="removal_frequency_submissions", 
                            main_title="Distribution Of Removal Percentages", 
                            main_color ='green',freq_var='rem_perc',
                             xlabel = '% Submissions Removed', ylabel ='Frequency',
                             x_axis_vector=False, 
                             bins = 50,histtype="bar", dpi = 300)

In [ ]:
S_grouped_deletions["rem_perc"].describe()

In [ ]:
print(S_grouped_deletions.to_string())

In [ ]:
#----------------------------#
# 2.)  Similarity to avg vector
#----------------------------#

In [ ]:
# I've computed this back then in subreddit_submission
embed_dict = np.load(data_path + 'embed_dict.npy',allow_pickle='TRUE').item()

In [ ]:
def get_subreddit_embeddings(c_S):
    #c_S: current subreddit
    local_embed_list = []
    id_list = c_S["id"].unique().tolist()
    [local_embed_list.append(embed_dict[x]) for x in id_list]
    return(local_embed_list)

In [ ]:
def avg_embedding(list_of_embs):
    # averages of vectors should be allowed IF this is a vector
    # https://www.tf.uni-kiel.de/matwis/amat/admat_en/kap_2/basics/b2_1_1.html
    # https://math.stackexchange.com/questions/80923/average-of-multiple-vectors
    out = list_of_embs[0]
    for i in range(1,len(list_of_embs)):
        out = out + list_of_embs[i]
    return(out/len(list_of_embs))

In [ ]:
#Old
#Get average embedding
#def avg_embedding_array(emb_matrix):
#    # averages of vectors should be allowed IF this is a vector
#    # https://www.tf.uni-kiel.de/matwis/amat/admat_en/kap_2/basics/b2_1_1.html
#    # https://math.stackexchange.com/questions/80923/average-of-multiple-vectors
#    out = emb_matrix[0,]
#    for i in range(1,emb_matrix.shape[0]):
#        out = out + emb_matrix[i,]
#    return(out/emb_matrix.shape[1])

In [ ]:
def reg_df_get(subreddit_string):     
    subreddit_df = S.loc[S["subreddit"] == subreddit_string,].copy()
    emb_list = get_subreddit_embeddings(subreddit_df)
    mean_emb  = avg_embedding(emb_list)
    sim_2_mean_vec = [np.inner(mean_emb, emb_list[i][0])[0] for i in range(0, len(emb_list))]
    #
    subreddit_df["similarity"] = 0
    subreddit_df["similarity"] = sim_2_mean_vec
    return(subreddit_df)

In [ ]:
corona_df = reg_df_get("Coronavirus")

In [ ]:
#del embed_dict
#del S

In [ ]:
#----------------------------#
# 3.) Statistics
#----------------------------#

In [ ]:
print(corona_df.loc[corona_df["removed"]==1,"similarity"].mean())
print(corona_df.loc[corona_df["removed"]==0,"similarity"].mean())

In [ ]:
import statsmodels.api as sm

In [ ]:
#run logit
predictors = sm.add_constant(corona_df["similarity"])
logit_model=sm.Logit(corona_df["removed"],predictors)
result=logit_model.fit()
print(result.summary2())

In [ ]:
AME = result.get_margeff(at = "mean",method="dydx")
print(AME.summary())
# A unit increase in similarity to the mean decreases the porbability to be deleted 
# by 80%

In [ ]:
#.)
# Probability at 1
print(np.exp(-1.1186 + -5.7975 )/(1+np.exp(-1.1186 + -5.7975 )))
#2.)
#Threshold:
#interc + similarity*threshold = 0
#thr = -intercept/similarit
print(-1.1186/float(-5.7975 ))
# If the threshold is lower  then this it will be predicted as kept
# Otherwise as deleted

In [ ]:
# Ok I want to find the Point at which the effect of distance on 
# Probability to being deleted is maximal
# I couldn't find the proper method to get the marginal effects at each point
# However following thi great link
#https://stackoverflow.com/questions/34439/finding-what-methods-a-python-object-has
# I could extract ALL methods with dir(AME)
# and margeff seems to do the job
AME = result.get_margeff(at = "all",method="dydx")
margeff_list = AME.margeff

In [ ]:
corona_df["ME"] = margeff_list

In [ ]:
simple_frequency(corona_df, path= plot_path, file_name="Marginal_Effects_Corona", 
                            main_title="Marginal Effects Frequency", 
                            main_color ='black',freq_var='ME',
                             xlabel = 'Marginal Effect', ylabel ='Frequency',
                             x_axis_vector=False, 
                             bins = 50,histtype="step", dpi = 300)

In [ ]:
result.predict().shape
# This link below didn't help :/
#https://stackoverflow.com/questions/26528019/python-statsmodel-api-logistic-regression-logit
#predicted = result.predict(corona_df["similarity"].values.T)
#predicted_choice = (predicted > threshold).astype(int)

In [ ]:
#Predict 
corona_df["PP"] = result.predict()

In [ ]:
simple_frequency(corona_df, path= plot_path, file_name="Predicted_Probabilities_Corona", 
                            main_title="Predicted Probabilites Frequency", 
                            main_color ='black',freq_var='PP',
                             xlabel = 'Predicted Probability', ylabel ='Frequency',
                             x_axis_vector=False, 
                             bins = 50,histtype="step", dpi = 300)

In [ ]:
# CONCLUSION:
# So the maximal probability to be deleted is 26 % 
# 

In [ ]:
#----------------------------#
# 4.) All
#----------------------------#

In [ ]:
threshold_list = []
effect_list = []
intercept_list = []
quant_05 = []
quant_25 = []
quant_50 = []
quant_75 = []
quant_95 = []
for s in S_grouped_deletions["subreddit"].tolist():
    submissions_df = reg_df_get(s)
    #
    predictors = sm.add_constant(submissions_df["similarity"])
    logit_model=sm.Logit(submissions_df["removed"],predictors)
    result=logit_model.fit()
    #
    threshold_list.append( - result.params[0]/float(result.params[1]))
    effect_list.append(result.params[1])
    intercept_list.append(result.params[0])
    #
    quant_05.append(submissions_df["similarity"].quantile(0.05))
    quant_25.append(submissions_df["similarity"].quantile(0.25))
    quant_50.append(submissions_df["similarity"].quantile(0.50))
    quant_75.append(submissions_df["similarity"].quantile(0.75))
    quant_95.append(submissions_df["similarity"].quantile(0.95))

In [ ]:
S_grouped_deletions

In [ ]:
S_grouped_deletions["intercept"] = intercept_list
S_grouped_deletions["effect"] = effect_list
S_grouped_deletions["threshold"] = threshold_list
#
S_grouped_deletions["quant_05"] = quant_05
S_grouped_deletions["quant_25"] = quant_25
S_grouped_deletions["quant_50"] = quant_50
S_grouped_deletions["quant_75"] = quant_75
S_grouped_deletions["quant_95"] = quant_95
S_grouped_deletions

In [ ]:
S_grouped_deletions.to_csv(path_or_buf=data_path + "S_grouped_deletions.csv", index =False)

In [ ]:
# Troubleshoot positive intercepts and effects
S_grouped_deletions.loc[(S_grouped_deletions["intercept"]>0) & 
                        (S_grouped_deletions["effect"]>0),]
# Vlad: Most get deleted. So of course avg is not a good thing
# And effect maybe low
# Or maybe redundnacy effect! Different Norms? 

In [ ]:
#
print(len(S_grouped_deletions.loc[(S_grouped_deletions["intercept"]<0) & 
                        (S_grouped_deletions["effect"]<0),]))
S_grouped_deletions.loc[(S_grouped_deletions["intercept"]<0) & 
                        (S_grouped_deletions["effect"]<0),].sort_values("effect")
# For example they really care about futureology, they don't want to posting other stuff
# You can catch the bubble! 
# But what if is negativ? wtf
#Vlad: thinking

In [ ]:
S_grouped_deletions.loc[S_grouped_deletions["effect"]>0,]

In [ ]:
S_grouped_deletions.loc[S_grouped_deletions["intercept"]>0,]

In [ ]:
S_grouped_deletions.loc[S_grouped_deletions["intercept"]<0,]

In [ ]:
S_grouped_deletions.loc[S_grouped_deletions["subreddit"]=="Coronavirus",]

In [ ]:
S_grouped_deletions["threshold"].describe()